# System Design for Data Engineers
## End-to-End Pipeline Design, Scalability & Interview Framework

**What you'll learn:**
- The system design interview framework (for DE-specific rounds)
- Requirements gathering: functional vs non-functional
- Scalability: horizontal/vertical, partitioning, sharding strategies
- Reliability: fault tolerance, idempotency, exactly-once, backpressure
- Data modeling for scale (denormalization, pre-aggregation, materialized views)
- Storage selection: OLTP vs OLAP vs lake vs stream
- 6 complete case studies designed step-by-step:
  1. Design a real-time analytics platform (Uber-style)
  2. Design a data lake for an e-commerce company
  3. Design a CDC pipeline (database replication)
  4. Design a recommendation data pipeline
  5. Design a metrics/observability platform
  6. Design a multi-tenant data platform
- Trade-off analysis and decision-making
- Interview questions and evaluation criteria

**Prerequisites:** 53_cs_fundamentals, 48_hadoop_ecosystem, 50_kafka, 56_aws
**Estimated Time:** 10-12 hours

---

> System design is THE differentiator for senior DE roles.
> It's not about memorizing architectures — it's about
> making INFORMED TRADE-OFFS and communicating them clearly.

---

---
# Section 1: The System Design Interview Framework

## The 4-Step Framework (45-60 minutes)

```
STEP 1: REQUIREMENTS (5-8 min)
  "Before I design anything, let me understand the requirements."
  
  Functional:
    - What data sources? (APIs, databases, files, streams)
    - What transformations? (aggregation, joins, enrichment)
    - What outputs? (dashboards, APIs, ML features, reports)
    - What queries will users run? (patterns matter for modeling)
  
  Non-Functional:
    - Scale: How much data? (GB/day? TB/day? PB total?)
    - Latency: How fresh? (real-time? minutes? hours? daily?)
    - Availability: What's the SLA? (99.9%? 99.99%?)
    - Durability: Can we lose data? (exactly-once vs at-least-once)
    - Cost: Any budget constraints?
    - Compliance: PII? GDPR? HIPAA?

STEP 2: HIGH-LEVEL DESIGN (10-15 min)
  "Here's my high-level architecture."
  
  Draw the major components:
    Sources → Ingestion → Storage → Processing → Serving
  
  Identify key technology choices:
    - Kafka for streaming? Batch from S3?
    - Delta Lake for storage? Redshift?
    - Spark for processing? Flink for real-time?
    - Airflow for orchestration?

STEP 3: DEEP DIVE (15-20 min)
  "Let me dive into [the most complex/interesting part]."
  
  Pick 1-2 components and go deep:
    - Data model (schema, partitioning, indexing)
    - Scalability (what happens at 10x, 100x scale?)
    - Failure handling (what if this component fails?)
    - Trade-offs (why THIS choice over alternatives?)

STEP 4: WRAP-UP (5-10 min)
  "Let me address operational concerns."
  
  - Monitoring & alerting
  - Cost estimation
  - Future evolution (what changes at 10x scale?)
  - Known limitations and how to mitigate
```

## What Interviewers Evaluate

| Criteria | What They Look For | Red Flag |
|----------|-------------------|----------|
| **Requirements** | Asks clarifying questions, doesn't assume | Jumps to solution immediately |
| **Breadth** | Covers all components end-to-end | Focuses on one piece only |
| **Depth** | Can deep-dive on any component when asked | Stays superficial |
| **Trade-offs** | Explains WHY, not just WHAT | "This is the best" without reasoning |
| **Scale** | Considers growth, failure modes | Design works for current scale only |
| **Communication** | Clear, structured, checks in with interviewer | Monologues without interaction |

---
# Section 2: System Design Building Blocks

## The Data Platform Stack

```
┌────────────────────────────────────────────────────────────┐
│ CONSUMPTION LAYER                                           │
│ Dashboards │ APIs │ ML Models │ Reverse ETL │ Data Products │
├────────────────────────────────────────────────────────────┤
│ SERVING LAYER                                               │
│ Data Warehouse │ Feature Store │ OLAP Cube │ Search Index   │
├────────────────────────────────────────────────────────────┤
│ PROCESSING LAYER                                            │
│ Batch (Spark) │ Stream (Flink/Kafka) │ Micro-batch (SS)    │
├────────────────────────────────────────────────────────────┤
│ STORAGE LAYER                                               │
│ Data Lake (S3+Delta) │ Data Warehouse │ Operational DB      │
├────────────────────────────────────────────────────────────┤
│ INGESTION LAYER                                             │
│ Batch (files, DB dumps) │ CDC (Debezium) │ Stream (Kafka)  │
├────────────────────────────────────────────────────────────┤
│ ORCHESTRATION │ CATALOG │ GOVERNANCE │ MONITORING │ QUALITY │
└────────────────────────────────────────────────────────────┘
```

## Storage Selection Guide

| Need | Technology | Why |
|------|-----------|-----|
| Transactional writes | PostgreSQL, MySQL | ACID, low latency |
| Analytical queries | Redshift, BigQuery, Databricks SQL | Columnar, MPP |
| Flexible schema, large scale | S3 + Delta Lake | Schema evolution, time travel |
| Key-value lookups (<5ms) | DynamoDB, Redis | Sub-millisecond at scale |
| Full-text search | Elasticsearch/OpenSearch | Inverted index |
| Time-series | TimescaleDB, InfluxDB, Timestream | Optimized for time-range queries |
| Graph relationships | Neo4j, Neptune | Traversal queries |
| ML features (real-time) | Redis, DynamoDB, Feature Store | Low-latency serving |

## Scalability Patterns

| Pattern | Description | Use When |
|---------|-------------|----------|
| **Horizontal partitioning** | Split data across nodes by key | Large tables (>100GB) |
| **Vertical partitioning** | Split columns (hot vs cold) | Wide tables with mixed access |
| **Read replicas** | Copies for read traffic | Read-heavy workloads |
| **Caching** | In-memory layer (Redis) | Repeated queries, <10ms needed |
| **Pre-aggregation** | Materialized rollups | Dashboard queries on huge data |
| **Lambda architecture** | Batch + speed layer | Need both historical + real-time |
| **Kappa architecture** | Stream-only (reprocess from log) | When streaming can handle all |
| **CQRS** | Separate read/write paths | Different optimization per path |

---
# Section 3: Case Study 1 — Real-Time Analytics Platform

## Problem Statement

> "Design a real-time analytics platform for a ride-sharing company (like Uber).
> The platform must show live metrics (rides in progress, driver supply/demand,
> surge pricing zones) and also support historical analysis."

## Step 1: Requirements

```
FUNCTIONAL:
  - Ingest ride events (request, match, start, end, cancel) — millions/day
  - Real-time dashboard: rides in progress by city, ETA, surge zones
  - Historical analytics: revenue by day/week, driver utilization, funnel
  - Alerts: supply/demand imbalance, fraud detection
  - ML: demand forecasting, dynamic pricing model inputs

NON-FUNCTIONAL:
  - Scale: 10M rides/day, 500K events/second at peak
  - Latency: Real-time dashboard < 5 seconds freshness
  - Historical queries: < 30 seconds for any ad-hoc query
  - Availability: 99.95% for dashboards
  - Retention: Raw events 90 days, aggregates 3 years
  - Regions: Multi-region (US, EU, APAC)
```

## Step 2: High-Level Architecture

```
RIDE EVENTS                    STREAMING                      SERVING
┌──────────┐                 ┌───────────────┐             ┌──────────────┐
│ Driver   │───────┐         │               │             │ REAL-TIME    │
│ App      │       │         │  KAFKA (MSK)  │────────────>│ Redis +      │
└──────────┘       │         │  Topics:       │             │ Pre-computed │
                   ▼         │  ride_events   │             │ aggregates   │
┌──────────┐  ┌─────────┐   │  driver_location│            └──────────────┘
│ Rider    │─>│ Event   │──>│  pricing_events│                    │
│ App      │  │ Gateway │   │               │             ┌──────────────┐
└──────────┘  │ (API)   │   └───────┬───────┘             │ DASHBOARDS   │
              └─────────┘           │                      │ (Grafana/    │
                                    │                      │  custom)     │
                          ┌─────────┼─────────┐            └──────────────┘
                          ▼         ▼         ▼
                   ┌──────────┐ ┌────────┐ ┌──────────┐   ┌──────────────┐
                   │ FLINK    │ │ FLINK  │ │ SPARK    │   │ HISTORICAL   │
                   │ (real-   │ │ (fraud │ │ (batch   │──>│ Redshift /   │
                   │  time    │ │  detect│ │  ETL)    │   │ Delta Lake   │
                   │  aggs)   │ │  tion) │ │          │   │              │
                   └────┬─────┘ └────────┘ └──────────┘   └──────────────┘
                        │                        │
                        ▼                        ▼
                   ┌──────────┐           ┌──────────────┐
                   │ Redis    │           │ S3 + Delta   │
                   │(live state│           │ (data lake)  │
                   │ per city) │           │              │
                   └──────────┘           └──────────────┘
```

## Step 3: Deep Dive — Real-Time Aggregation

```
FLINK JOB: Live Metrics per City (5-second windows)

Input: ride_events topic
Processing:
  - KeyBy: city_id
  - Window: TumblingEventTime(5 seconds)
  - Aggregate per window:
    - active_rides_count
    - avg_eta_seconds
    - supply_demand_ratio (available_drivers / ride_requests)
    - surge_multiplier (if demand > 2x supply)
Output: Write to Redis (city_id → metrics JSON, TTL 30s)

Dashboard reads from Redis every 5 seconds.
If Redis is down: fallback to Kafka consumer (slightly stale).
```

## Step 4: Trade-offs Discussion

| Decision | Choice | Alternative | Why |
|----------|--------|-------------|-----|
| Streaming engine | Flink | Spark Structured Streaming | Sub-second latency needed |
| Real-time store | Redis | DynamoDB | Lower latency, simpler for hot data |
| Historical store | Delta Lake | Redshift | Schema flexibility, cost at scale |
| Message broker | Kafka (MSK) | Kinesis | Higher throughput, replay capability |
| Orchestration | Airflow (batch) + Flink (stream) | All Flink | Batch still needed for backfills |

---
# Section 4: Case Study 2 — E-Commerce Data Lake

## Problem Statement

> "Design a data lake for a mid-size e-commerce company (50M orders/year).
> Support BI reporting, data science, and operational analytics."

## Architecture

```
SOURCES                        DATA LAKE (S3 + Delta)           CONSUMERS
┌──────────┐                  ┌────────────────────────────┐   ┌──────────┐
│PostgreSQL │──CDC──>         │ BRONZE        SILVER  GOLD  │   │ Looker   │
│(orders,   │(Debezium)│     │ (raw, as-is) (clean) (biz) │   │ (BI)     │
│ customers)│          │     │                              │   └──────────┘
└──────────┘          │     │  orders_raw → orders  → daily │
                      ▼     │  clicks_raw → sessions→ funnel│   ┌──────────┐
┌──────────┐    ┌─────────┐ │  products   → catalog → reco  │   │Data Science│
│Clickstream│──>│  KAFKA  │─>│                              │──>│(notebooks)│
│(events)   │   └─────────┘ │  Partitioned by date          │   └──────────┘
└──────────┘                 │  Delta Lake (ACID, versioned) │
                             │  Unity Catalog (governance)   │   ┌──────────┐
┌──────────┐    ┌─────────┐ │                              │   │Operational│
│3rd party │──>│ Airflow │─>│  Orchestration: Airflow       │──>│ (alerts) │
│APIs      │   │(scheduled│  │  Quality: Great Expectations  │   └──────────┘
│(Stripe,  │   │ ingestion)│ │  Catalog: Unity Catalog       │
│ Shopify) │   └─────────┘ └────────────────────────────────┘
└──────────┘

KEY DESIGN DECISIONS:
  1. CDC (not batch dump): minimize source DB impact, near-real-time
  2. Medallion architecture: separate raw from business logic
  3. Delta Lake: ACID, schema enforcement, time travel
  4. Partition by date: 99% of queries filter by time range
  5. Unity Catalog: column-level security for PII
  6. Separate compute + storage: scale independently
```

## Data Model (Gold Layer)

```sql
-- Fact: orders (grain = one order)
gold.fact_orders (
    order_id, order_date, customer_id, product_id,
    quantity, revenue, discount, shipping_cost,
    payment_method, channel, status,
    -- Slowly changing dimension keys:
    customer_dim_key, product_dim_key
)
PARTITIONED BY (order_date)
CLUSTERED BY (customer_id)  -- Liquid Clustering

-- Dimension: customers (SCD Type 2)
gold.dim_customers (
    customer_dim_key, customer_id,
    name, email_masked, segment, city, state,
    effective_from, effective_to, is_current
)

-- Pre-aggregated: daily metrics (for fast dashboard queries)
gold.daily_metrics (
    date, segment, channel,
    orders, revenue, unique_customers, avg_order_value,
    new_customers, returning_customers
)
```

---
# Section 5: Case Study 3 — CDC Pipeline Design

## Problem Statement

> "Design a Change Data Capture pipeline that replicates data from 50 PostgreSQL
> tables to a data lake in near-real-time (< 5 minute latency)."

## Architecture

```
PostgreSQL (Source)         CDC Pipeline              Data Lake (Target)
┌──────────────┐         ┌──────────────────┐      ┌──────────────────┐
│              │         │                  │      │                  │
│  WAL (Write │──────>  │  DEBEZIUM        │      │  S3 + Delta Lake │
│  Ahead Log) │         │  (CDC connector) │      │                  │
│              │         │                  │      │  Bronze:         │
│  50 tables   │         │  Reads WAL       │      │  raw CDC events  │
│  ~100K       │         │  Converts to     │      │  (insert/update/ │
│  changes/min │         │  Kafka events    │      │   delete + before/│
│              │         │                  │      │   after images)  │
└──────────────┘         └────────┬─────────┘      │                  │
                                  │                 │  Silver:         │
                         ┌────────▼─────────┐      │  Current state   │
                         │  KAFKA (MSK)     │      │  (MERGE applied) │
                         │                  │      │                  │
                         │  Topic per table:│      │  Gold:           │
                         │  cdc.orders      │──────>  Business models  │
                         │  cdc.customers   │      │                  │
                         │  cdc.products    │      └──────────────────┘
                         │  ...             │
                         └──────────────────┘
                                  │
                         ┌────────▼─────────┐
                         │  SPARK STREAMING │
                         │  (micro-batch)   │
                         │                  │
                         │  Read from Kafka │
                         │  Apply MERGE to  │
                         │  Delta Lake      │
                         │  (upsert/delete) │
                         └──────────────────┘

MERGE LOGIC:
  MERGE INTO silver.customers AS target
  USING (batch of CDC events) AS source
  ON target.customer_id = source.customer_id
  WHEN MATCHED AND source.op = 'u' THEN UPDATE SET ...
  WHEN MATCHED AND source.op = 'd' THEN DELETE
  WHEN NOT MATCHED AND source.op IN ('c', 'r') THEN INSERT ...
```

## Key Design Considerations

| Challenge | Solution |
|-----------|----------|
| Schema evolution (source adds column) | Schema Registry + Spark schema merge |
| Out-of-order events | Event timestamp ordering within MERGE |
| Initial load (50 tables, first time) | Snapshot mode → switch to streaming |
| Large tables (100M+ rows) | Parallel initial load, then incremental |
| Delete handling | Soft delete in silver (is_deleted flag) + hard delete in gold |
| Exactly-once delivery | Kafka offset tracking + idempotent MERGE |
| Monitoring | Track lag per table, alert if > 5 min behind |

In [ ]:
# Additional case study summaries
print("ADDITIONAL SYSTEM DESIGN CASE STUDIES")
print("=" * 60)

print("""
CASE STUDY 4: RECOMMENDATION DATA PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Requirements:
  - User activity events (views, clicks, purchases) → feature pipeline
  - Compute user-item interaction features (real-time + batch)
  - Serve features to recommendation model (<50ms)

Architecture:
  Events → Kafka → Flink (real-time features) → Redis (online store)
                 → Spark (batch features) → Delta Lake (offline store)
  Model serving reads from Redis at inference time.

Key decisions:
  - Two-speed features: real-time (last 30 min activity) + batch (historical)
  - Feature store bridges offline training and online serving
  - Item embeddings pre-computed daily, user embeddings real-time

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CASE STUDY 5: METRICS / OBSERVABILITY PLATFORM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Requirements:
  - Collect metrics from 1000+ microservices (CPU, latency, errors)
  - 1M metrics/second ingest rate
  - Query: last 1 hour (fast), last 30 days (medium), last 1 year (slow)

Architecture:
  Services → StatsD/OTel → Kafka → Flink (pre-aggregate) → 
    Hot: Redis (last 1 hour, 1-second resolution)
    Warm: TimescaleDB (last 30 days, 1-minute resolution)
    Cold: S3/Parquet (full history, 5-minute resolution)
  
  Grafana reads from appropriate tier based on time range.
  Pre-aggregate at ingest to reduce query-time work.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CASE STUDY 6: MULTI-TENANT DATA PLATFORM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Requirements:
  - 50 internal teams share one data platform
  - Isolation: team A can't see team B's data
  - Self-service: teams can create their own pipelines
  - Cost allocation: bill each team for what they use

Architecture:
  Shared infrastructure:
    - Unity Catalog (namespaces per team)
    - Shared Kafka cluster (ACLs per topic)
    - Shared Airflow (pools + queue per team)
    - Shared compute (Kubernetes namespaces with quotas)
  
  Isolation mechanisms:
    - Unity Catalog: catalog.schema.table ownership
    - Row-level security for shared tables
    - Network: VPC per sensitivity level
    - Compute: resource quotas (max DPUs per team)
  
  Cost: tag all resources, aggregate by team in billing dashboard.
""")

In [ ]:
# System design interview tips
print("\nSYSTEM DESIGN INTERVIEW TIPS")
print("=" * 60)

print("""
DO:
  ✓ Ask clarifying questions first (2-3 minutes minimum)
  ✓ State assumptions explicitly ("I'll assume 10TB/day")
  ✓ Draw architecture diagram (even in text form)
  ✓ Discuss trade-offs ("I chose X over Y because...")
  ✓ Consider failure modes ("What if Kafka goes down?")
  ✓ Mention monitoring ("I'd alert on consumer lag > 5 min")
  ✓ Quantify: throughput, latency, storage estimates
  ✓ Check in with interviewer ("Should I dive deeper here?")

DON'T:
  ✗ Jump to technology without understanding requirements
  ✗ Design for Google scale when it's a startup
  ✗ Ignore cost ("That Flink cluster costs $50K/month")
  ✗ Forget operational concerns (deployment, monitoring)
  ✗ Over-engineer (simple is better until proven otherwise)
  ✗ Use buzzwords without understanding ("Let's use blockchain")
  ✗ Ignore the interviewer's hints/questions

BACK-OF-ENVELOPE CALCULATIONS:
  - 1M events/day = ~12 events/second (not that much)
  - 1B events/day = ~12K events/second (need Kafka + Flink)
  - 1 KB per event × 1M/day = ~1 GB/day raw
  - 1 KB per event × 1B/day = ~1 TB/day raw
  - Parquet compression: ~5-10x → 1 TB raw → 100-200 GB stored
  - S3 cost: 1 TB stored = ~$23/month
  - Spark processing: 1 TB takes ~10-30 minutes (depending on transforms)
  - Kafka throughput: 1 partition ≈ 10 MB/s ≈ 10K events/s (at 1KB/event)
""")

print("COMMON FOLLOW-UP QUESTIONS:")
followups = [
    "What happens if your Kafka cluster goes down?",
    "How would you handle 10x growth in data volume?",
    "How do you ensure exactly-once processing?",
    "How would you handle schema evolution?",
    "What's your data quality strategy?",
    "How do you estimate the cost of this architecture?",
    "How would you test this system end-to-end?",
    "What would you monitor and alert on?",
]
for i, q in enumerate(followups, 1):
    print(f"  {i}. {q}")

---
# Summary: System Design Cheat Sheet

## The Framework (memorize this structure)

1. **Requirements** (5 min): Functional + Non-functional + Scale numbers
2. **High-Level Design** (10 min): Sources → Ingest → Store → Process → Serve
3. **Deep Dive** (20 min): Data model, scalability, failure handling
4. **Operations** (5 min): Monitoring, cost, evolution

## Quick Architecture Patterns

| Pattern | When | Components |
|---------|------|-----------|
| **Batch ETL** | Daily reports, overnight processing | Airflow + Spark + Delta Lake |
| **Real-time** | <5s freshness, live dashboards | Kafka + Flink + Redis |
| **Lambda** | Need both real-time + historical | Kafka + Flink (speed) + Spark (batch) |
| **Kappa** | Can reprocess from stream | Kafka (retain all) + Flink |
| **CDC** | Database replication, near-real-time | Debezium + Kafka + Spark MERGE |
| **Event sourcing** | Full audit trail, replay | Kafka (compacted) + materialized views |
| **Lakehouse** | Unified batch + interactive | Delta Lake + Spark + Databricks SQL |

## Capacity Planning Quick Math

| Metric | Estimation |
|--------|-----------|
| Events/day → events/sec | ÷ 86,400 |
| 1 event = 1 KB (typical) | Adjust per use case |
| Raw → Parquet (compressed) | ÷ 5-10x |
| Kafka partitions needed | events/sec ÷ 10K per partition |
| Spark workers for 1 TB | ~10-20 workers (4 cores each) |
| S3 cost | $23/TB/month (Standard) |
| Redshift cost | ~$0.25/node/hour (ra3.xlplus) |

---
*System Design for Data Engineers*